In [ ]:
import numpy as np
import scipy.ndimage as ndimage

# =====================================================================
# 1. PURELY GEOMETRIC ANISOTROPIC NON-GAUSSIAN FIELD GENERATOR
# =====================================================================

def generate_falsification_field(grid_size=128, seed=42):
    """
    Constructs a 3D scalar field breaking both Gaussianity and spatial isotropy.
    - No physics, no Hamiltonians, no Ising spins.
    - Uses heavy-tailed Laplace noise.
    - Implements asymmetric correlation lengths: xi_x >> xi_y >> xi_z
    - Embeds procedural geometric structures (sheets and directional filaments).
    """
    np.random.seed(seed)

    # A. Generate Heavy-Tailed (Non-Gaussian) Laplace Noise Baseline
    # This introduces a distribution dominated by extreme outliers
    delta = np.random.laplace(0, 1.0, size=(grid_size, grid_size, grid_size))

    # B. Inject Asymmetric Correlation Lengths (Directional Shearing)
    # x-axis: Highly extended (long correlation)
    # y-axis: Moderately extended
    # z-axis: Sharp, localized fluctuations (short correlation)
    sigma_x, sigma_y, sigma_z = 5.0, 2.0, 0.5
    delta = ndimage.gaussian_filter(delta, sigma=(sigma_x, sigma_y, sigma_z))

    # C. Procedural Geometry Injection (Non-Equilibrium Broken Symmetry)
    x, y, z = np.indices((grid_size, grid_size, grid_size))

    # 1. Sheet structures parallel to the X-Y plane at asymmetric intervals
    sheets = np.sin(4.0 * np.pi * z / grid_size) > 0.8
    delta[sheets] += 2.5

    # 2. Filament structures directed strictly along the X-axis
    filaments = (np.sin(8.0 * np.pi * y / grid_size) > 0.9) & (np.cos(8.0 * np.pi * z / grid_size) > 0.9)
    delta[filaments] += 3.0

    # Standardize to bring variance to unity without changing geometry
    delta = (delta - np.mean(delta)) / (np.std(delta) + 1e-10)
    return delta

# =====================================================================
# 2. EXPLICIT PLACEHOLDER OPERATORS FOR REPLACEMENT
# =====================================================================

def Thinning_Operator(field):
    """
    R_info: Coarse-grains spatial volume down via 2x2x2 block projection.
    Placeholder version uses local coordinate pooling with downsampling.
    """
    # Dynamic scale adaptation relative to current grid volume
    current_L = field.shape[0]
    smoothing = max(0.5, 0.5 * (current_L / 128.0))
    smoothed = ndimage.gaussian_filter(field, sigma=smoothing)

    # Hard 2x2x2 block downsample slicing
    return smoothed[::2, ::2, ::2]

def ACCR_Operator(field):
    """
    R_consist: Macro-consistency checker and structural feature protection.
    Placeholder version enforces global mass conservation and local peak reinforcement.
    """
    # Enforce zero-mean mass conservation across the scale jump
    field_macro = field - np.mean(field)
    local_std = np.std(field_macro)

    # Asymmetric feature-preservation nudge (Multiplicative variant rules)
    # Protects high-density peaks from being dissolved during downsampling
    if local_std > 0.1:
        field_macro = np.where(field_macro > 0.4 * local_std, field_macro * 1.15, field_macro * 0.85)

    return ndimage.gaussian_filter(field_macro, sigma=0.3)

# =====================================================================
# 3. SPATIAL GEOMETRIC DIAGNOSTICS (NO PHYSICS)
# =====================================================================

def compute_directional_correlation_lengths(field):
    """
    Measures the 1/e decay length of spatial autocorrelation along x, y, and z axes
    to track how spatial anisotropy evolves under the RG flow.
    """
    L = field.shape[0]
    f_arr = field - np.mean(field)
    f_fft = np.fft.fftn(f_arr)
    spatial_corr = np.fft.ifftn(np.abs(f_fft)**2).real
    spatial_corr = np.fft.fftshift(spatial_corr) / (np.max(spatial_corr) + 1e-10)

    mid = L // 2

    # Extract ID axis-cut profiles from the center of the 3D correlation box
    profile_x = spatial_corr[:, mid, mid]
    profile_y = spatial_corr[mid, :, mid]
    profile_z = spatial_corr[mid, mid, :]

    xi_axes = []
    for profile in [profile_x, profile_y, profile_z]:
        half_profile = profile[mid:]
        indices = np.where(half_profile < (1.0 / np.e))[0]
        xi = indices[0] if len(indices) > 0 else 1.0
        xi_axes.append(max(float(xi), 1.0))

    return xi_axes[0], xi_axes[1], xi_axes[2]

def extract_and_measure_filaments(field):
    """
    Skeletons the field using spatial percentiles to compute macro-structural
    correlation length (xi_fil) and invariant ratios.
    """
    L = field.shape[0]

    # Isolate high-density geometric structural ridges
    skeleton = field > np.percentile(field, 75)

    # Compute isotropic radial decay profile
    f_arr = skeleton.astype(float) - np.mean(skeleton)
    corr = np.fft.fftshift(np.fft.ifftn(np.abs(np.fft.fftn(f_arr))**2).real)
    corr /= (np.max(corr) + 1e-10)

    mid = L // 2
    x, y, z = np.indices((L, L, L))
    r = np.sqrt((x-mid)**2 + (y-mid)**2 + (z-mid)**2).flatten()
    c = corr.flatten()

    bins = np.arange(0, mid, 1)
    profile = []
    for b in range(len(bins)-1):
        mask = (r >= bins[b]) & (r < bins[b+1])
        profile.append(np.mean(c[mask]) if np.any(mask) else 0.0)

    profile = np.nan_to_num(np.array(profile))
    indices = np.where(profile < (1.0 / np.e))[0]

    xi_fil = bins[indices[0]] if len(indices) > 0 else 1.5
    return max(float(xi_fil), 1.5)

# =====================================================================
# 4. TAPS RUNTIME FLOW AND STRESS-TEST EVALUATION
# =====================================================================

print("=" * 82)
print("              AC-CR GEOMETRIC FALSIFICATION ENGINE: 3D ANISOTROPIC FLOW       ")
print("=" * 82)
print(f"{'Scale (L)':<10} | {'xi_x':<6} | {'xi_y':<6} | {'xi_z':<6} | {'xi_fil':<8} | {'Invariant Ratio (xi_fil/L)':<24}")
print("-" * 82)

# Initialize the broken-symmetry test field
delta = generate_falsification_field(grid_size=128, seed=42)
history = []

# Iterated scale reduction flow loop
for step in range(4):
    L = delta.shape[0]

    # 1. Execute spatial diagnostics
    xi_x, xi_y, xi_z = compute_directional_correlation_lengths(delta)
    xi_fil = extract_and_measure_filaments(delta)
    ratio = xi_fil / L
    history.append(ratio)

    # 2. Output instant data slice
    print(f"L = {L:3d}    | {xi_x:5.1f} | {xi_y:5.1f} | {xi_z:5.1f} | {xi_fil:7.2f} | {ratio:.4f}")

    # 3. Apply the operator cascade to downsample the block volume
    if L > 16:
        delta = Thinning_Operator(delta)
        delta = ACCR_Operator(delta)

print("=" * 82)

# 5. AUTOMATED FALSIFICATION VERDICT
max_delta_ratio = np.max(history) - np.min(history)
print("\n--- STRESS TEST EVALUATION ---")
print(f"Maximum Variant Variance Track Drift: {max_delta_ratio:.4f}")

if max_delta_ratio < 0.05:
    print("VERDICT: SURVIVED. The invariant ratio holds steady despite extreme spatial asymmetry.")
    print("Interpretation: AC-CR functions successfully as a robust geometric calculus.")
else:
    print("VERDICT: FALSIFIED. The invariant ratio diverged under spatial directional shearing.")
    print("Interpretation: Framework requires structural connectivity constraints to stabilize.")
print("=" * 82)

              AC-CR GEOMETRIC FALSIFICATION ENGINE: 3D ANISOTROPIC FLOW       
Scale (L)  | xi_x   | xi_y   | xi_z   | xi_fil   | Invariant Ratio (xi_fil/L)
----------------------------------------------------------------------------------
L = 128    |   1.0 |   1.0 |   6.0 |   11.00 | 0.0859
L =  64    |   1.0 |   1.0 |   3.0 |    6.00 | 0.0938
L =  32    |   1.0 |   1.0 |   2.0 |    3.00 | 0.0938
L =  16    |   1.0 |   1.0 |   1.0 |    1.50 | 0.0938

--- STRESS TEST EVALUATION ---
Maximum Variant Variance Track Drift: 0.0078
VERDICT: SURVIVED. The invariant ratio holds steady despite extreme spatial asymmetry.
Interpretation: AC-CR functions successfully as a robust geometric calculus.


In [ ]:
import numpy as np
import scipy.ndimage as ndimage

# =====================================================================
# 1. ADVERSARIAL FRACTAL, NON-STATIONARY, ROTATING FIELD GENERATOR
# =====================================================================

def generate_adversarial_field(grid_size=256, seed=101):
    """
    Constructs a 3D scalar field designed to destroy naïve statistical filters.
    - Zero physics equations, zero equilibrium conditions.
    - Non-Stationary: Heavy-tailed Laplace in the upper volume, Gaussian in the lower volume.
    - Fractal: Multi-scale fractional Brownian noise overlay.
    - Discontinuous: Hard jump boundaries cutting across the space.
    - Topological Defects: Randomly oriented embedded sheets, loops, and void regions.
    """
    np.random.seed(seed)
    x, y, z = np.indices((grid_size, grid_size, grid_size))
    mid = grid_size // 2

    # A. Non-Stationary Background Generation
    field = np.zeros((grid_size, grid_size, grid_size))
    field[:mid, :, :] = np.random.laplace(0.0, 1.5, size=(mid, grid_size, grid_size)) # Upper: Heavy-tailed
    field[mid:, :, :] = np.random.normal(0.0, 1.0, size=(grid_size - mid, grid_size, grid_size)) # Lower: Gaussian

    # B. Fractal Noise Overlay (Multi-scale fBm proxy via harmonic summing)
    # Generates power-law spatial roughness across the volume
    for frequency, weight in zip([2, 4, 8, 16], [1.0, 0.5, 0.25, 0.125]):
        noise = np.random.normal(0, 1, (grid_size, grid_size, grid_size))
        smoothed_noise = ndimage.gaussian_filter(noise, sigma=grid_size / (2.0 * frequency))
        field += smoothed_noise * weight

    # C. Sharp Discontinuities & Topological Defects
    # 1. Jump Surface: Hard boundary cleavage along an oblique plane
    jump_mask = (x + y + z) % (grid_size // 2) < (grid_size // 8)
    field[jump_mask] += 3.5

    # 2. Filament Loops: Toroidal high-density defect channels in random orientations
    r_torus = grid_size // 4
    r_tube = grid_size // 16
    torus_mask = (np.abs(np.sqrt((x - mid)**2 + (y - mid)**2) - r_torus) < r_tube) & (np.abs(z - mid) < r_tube)
    field[torus_mask] += 4.0

    # 3. Macro-Voids: Spherical regions of absolute zero information
    void_mask = (x - mid//2)**2 + (y - mid//2)**2 + (z - mid//2)**2 < (grid_size // 10)**2
    field[void_mask] = -5.0

    # Standardize field mapping
    field = (field - np.mean(field)) / (np.std(field) + 1e-10)
    return field

def apply_random_3d_rotation(field, angles=None, seed=None):
    """
    Rotates the 3D grid in space along two arbitrary axes.
    Used at each step to scramble spatial coordinates and test rotational invariance.
    """
    if seed is not None: np.random.seed(seed)
    if angles is None:
        angles = np.random.uniform(15.0, 75.0, size=2)

    # Rotate along the X-Y plane, then the Y-Z plane
    rotated = ndimage.rotate(field, angles[0], axes=(0, 1), reshape=False, mode='wrap')
    rotated = ndimage.rotate(rotated, angles[1], axes=(1, 2), reshape=False, mode='wrap')
    return rotated

# =====================================================================
# 2. EXPLICIT REPLACEABLE ADAPTIVE CALCULUS OPERATORS
# =====================================================================

def Thinning_Operator(field):
    """
    R_info: Volumetric downsampling projection step.
    Combines scale-proportional Gaussian footprint smoothing with a 2x2x2 stride slice.
    """
    current_L = field.shape[0]
    # Scale-adaptive smoothing footprint to isolate macro-features
    smoothing_scale = max(0.5, 0.5 * (current_L / 256.0))
    smoothed = ndimage.gaussian_filter(field, sigma=smoothing_scale)
    return smoothed[::2, ::2, ::2]

def ACCR_Operator(field):
    """
    R_consist: Non-linear macro-consistency structural resolver.
    Enforces total volume zero-mean constraints and applies an asymmetric peak edge trigger.
    """
    field_macro = field - np.mean(field)
    local_variance = np.std(field_macro)

    # Non-linear conditional structure protection rule
    # High-density features must actively push away from the background mean to resist smoothing
    if local_variance > 0.05:
        field_macro = np.where(field_macro > 0.5 * local_variance, field_macro * 1.20, field_macro * 0.80)

    return ndimage.gaussian_filter(field_macro, sigma=0.25)

# =====================================================================
# 3. ADVANCED SPATIAL SPATIAL-METRIC EVALUATION SUITE
# =====================================================================

def compute_spatial_metrics(field):
    """
    Computes directional correlation lengths (xi_x, xi_y, xi_z) via 1/e autocorrelation decay,
    and isolates macro-skeleton structures using a percentile density trigger to determine xi_fil.
    """
    L = field.shape[0]
    mid = L // 2

    # 1. Directional Axis Analysis via FFT
    f_arr = field - np.mean(field)
    spatial_corr = np.fft.ifftn(np.abs(np.fft.fftn(f_arr))**2).real
    spatial_corr = np.fft.fftshift(spatial_corr) / (np.max(spatial_corr) + 1e-10)

    xi_axes = []
    for axis_profile in [spatial_corr[:, mid, mid], spatial_corr[mid, :, mid], spatial_corr[mid, mid, :]]:
        half = axis_profile[mid:]
        idx = np.where(half < 0.3678)[0] # 1/e threshold
        xi_axes.append(max(float(idx[0]), 1.0) if len(idx) > 0 else 1.0)

    # 2. Isotropic Macro-Skeleton Analysis
    skeleton = field > np.percentile(field, 80)
    sk_arr = skeleton.astype(float) - np.mean(skeleton)
    sk_corr = np.fft.fftshift(np.fft.ifftn(np.abs(np.fft.fftn(sk_arr))**2).real)
    sk_corr /= (np.max(sk_corr) + 1e-10)

    x, y, z = np.indices((L, L, L))
    r = np.sqrt((x-mid)**2 + (y-mid)**2 + (z-mid)**2).flatten()
    c = sk_corr.flatten()

    bins = np.arange(0, mid, 1)
    profile = [np.mean(c[(r >= bins[b]) & (r < bins[b+1])]) for b in range(len(bins)-1)]
    profile = np.nan_to_num(np.array(profile))

    idx_fil = np.where(profile < 0.3678)[0]
    xi_fil = max(float(bins[idx_fil[0]]), 1.5) if len(idx_fil) > 0 else 1.5

    return xi_axes[0], xi_axes[1], xi_axes[2], xi_fil

# =====================================================================
# 4. ITERATED RUNTIME CASCADE TRACK (L = 256 -> 16)
# =====================================================================

print("=" * 95)
print("             AC-CR DESTRUCTIVE FALSIFICATION ENGINE: ADVERSARIAL SCALE FLOW          ")
print("=" * 95)
print(f"{'Scale (L)':<10} | {'xi_x':<6} | {'xi_y':<6} | {'xi_z':<6} | {'xi_fil':<8} | {'Ratio (xi_fil/L)':<18} | {'Rot. Variance (d_xi)':<20}")
print("-" * 95)

# Initialize the highly non-stationary adversarial landscape
delta = generate_adversarial_field(grid_size=256, seed=101)
ratio_history = []
rotation_variances = []

for step in range(5):
    L = delta.shape[0]

    # Measure baseline spatial metrics before rotation disturbance
    xi_x, xi_y, xi_z, xi_fil_pre = compute_spatial_metrics(delta)
    ratio_pre = xi_fil_pre / L

    # Execute structural scrambling: Rotate the 3D matrix arbitrarily before processing operators
    delta_rotated = apply_random_3d_rotation(delta, seed=42+step)
    _, _, _, xi_fil_post = compute_spatial_metrics(delta_rotated)

    # Record metrics
    rot_variance = abs(xi_fil_pre - xi_fil_post)
    rotation_variances.append(rot_variance)
    ratio_history.append(ratio_pre)

    print(f"L = {L:3d}    | {xi_x:5.1f} | {xi_y:5.1f} | {xi_z:5.1f} | {xi_fil_pre:7.2f} | {ratio_pre:.4f}           | {rot_variance:.4f}")

    # Apply downsampling cascade for the next scale boundary step
    if L > 16:
        delta = Thinning_Operator(delta_rotated) # Pass the scrambled field into the operators
        delta = ACCR_Operator(delta)

print("=" * 95)

# 5. AUTOMATED DESTRUCTIVE VERDICT ANALYSIS
max_ratio_drift = np.max(ratio_history) - np.min(ratio_history)
mean_rot_variance = np.mean(rotation_variances)

print("\n--- ADVERSARIAL STRESS TEST ANALYSIS ---")
print(f"Maximum Scaling Ratio Drift Track: {max_ratio_drift:.4f}")
print(f"Mean Rotational Scale Variance:    {mean_rot_variance:.4f}")

# Strict Falsification Thresholds
# If ratio drift exceeds 5% or rotation causes massive scale jumps, the calculus breaks.
if max_ratio_drift < 0.05 and mean_rot_variance <= 1.0:
    print("\nVERDICT: SURVIVED.")
    print("Interpretation: The Thinning + AC-CR operators demonstrate genuine scale-agnostic")
    print("and rotation-covariant geometric stability, overcoming explicit statistical non-stationarity.")
else:
    print("\nVERDICT: FALSIFIED.")
    print("Interpretation: The pipeline broke under the adversarial layout. The operators are dependent")
    print("on coordinate alignment or statistical homogeneity, failing as a general geometric calculus.")
print("=" * 95)

             AC-CR DESTRUCTIVE FALSIFICATION ENGINE: ADVERSARIAL SCALE FLOW          
Scale (L)  | xi_x   | xi_y   | xi_z   | xi_fil   | Ratio (xi_fil/L)   | Rot. Variance (d_xi)
-----------------------------------------------------------------------------------------------
L = 256    |   7.0 |   7.0 |   7.0 |    7.00 | 0.0273           | 2.0000
L = 128    |  24.0 |  13.0 |   5.0 |    6.00 | 0.0469           | 0.0000
L =  64    |   8.0 |   3.0 |   6.0 |    3.00 | 0.0469           | 0.0000
L =  32    |   2.0 |   4.0 |   4.0 |    2.00 | 0.0625           | 0.0000
L =  16    |   2.0 |   2.0 |   1.0 |    1.50 | 0.0938           | 0.0000

--- ADVERSARIAL STRESS TEST ANALYSIS ---
Maximum Scaling Ratio Drift Track: 0.0664
Mean Rotational Scale Variance:    0.4000

VERDICT: FALSIFIED.
Interpretation: The pipeline broke under the adversarial layout. The operators are dependent
on coordinate alignment or statistical homogeneity, failing as a general geometric calculus.


In [ ]:
import numpy as np
import scipy.ndimage as ndimage
from scipy.spatial.distance import cdist

# =====================================================================
# 1. THE ADVERSARIAL GENERATOR (RETAINED FOR STRICT COMPARISON)
# =====================================================================

def generate_adversarial_field(grid_size=256, seed=101):
    """Retains the exact hostile non-stationary, fractal, and defective landscape."""
    np.random.seed(seed)
    x, y, z = np.indices((grid_size, grid_size, grid_size))
    mid = grid_size // 2

    field = np.zeros((grid_size, grid_size, grid_size))
    field[:mid, :, :] = np.random.laplace(0.0, 1.5, size=(mid, grid_size, grid_size))
    field[mid:, :, :] = np.random.normal(0.0, 1.0, size=(grid_size - mid, grid_size, grid_size))

    for frequency, weight in zip([2, 4, 8, 16], [1.0, 0.5, 0.25, 0.125]):
        noise = np.random.normal(0, 1, (grid_size, grid_size, grid_size))
        smoothed_noise = ndimage.gaussian_filter(noise, sigma=grid_size / (2.0 * frequency))
        field += smoothed_noise * weight

    jump_mask = (x + y + z) % (grid_size // 2) < (grid_size // 8)
    field[jump_mask] += 3.5

    r_torus, r_tube = grid_size // 4, grid_size // 16
    torus_mask = (np.abs(np.sqrt((x - mid)**2 + (y - mid)**2) - r_torus) < r_tube) & (np.abs(z - mid) < r_tube)
    field[torus_mask] += 4.0

    void_mask = (x - mid//2)**2 + (y - mid//2)**2 + (z - mid//2)**2 < (grid_size // 10)**2
    field[void_mask] = -5.0

    return (field - np.mean(field)) / (np.std(field) + 1e-10)

def apply_random_3d_rotation(field, seed=None):
    """Applies oblique geometric re-orientations to challenge rotational invariance."""
    if seed is not None: np.random.seed(seed)
    angles = np.random.uniform(15.0, 75.0, size=2)
    rotated = ndimage.rotate(field, angles[0], axes=(0, 1), reshape=False, mode='wrap')
    return ndimage.rotate(rotated, angles[1], axes=(1, 2), reshape=False, mode='wrap')

# =====================================================================
# 2. COORDINATE-FREE GEOMETRIC DIAGNOSTICS (V2 SUITE)
# =====================================================================

def compute_coordinate_free_metrics(field, sample_size=1000):
    """
    Evaluates spatial structure without projecting onto x, y, or z axes.
    Uses point-pair statistics sampled from high-density features.
    """
    L = field.shape[0]

    # Isolate macro-structural features via a threshold profile
    threshold = np.percentile(field, 80)
    feature_indices = np.argwhere(field > threshold)

    if len(feature_indices) < 2:
        return 1.5, 1.5

    # Sub-sample points randomly to maintain high velocity and low memory footprints
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(feature_indices), size=min(sample_size, len(feature_indices)), replace=False)
    pts = feature_indices[sample_idx]

    # Generate the pairwise distance matrix: D_ij = ||x_i - x_j||
    dists = cdist(pts, pts, metric='euclidean')
    tri_dists = dists[np.triu_indices(dists.shape[0], k=1)]

    # 1. Compute xi_iso via the 1/e scale of the radial distribution profile
    counts, bin_edges = np.histogram(tri_dists, bins=30, range=(0, L // 2))
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    # Normalize density profile by volume elements of spherical shells
    shell_volumes = 4.0 * np.pi * (bin_centers**2) * (bin_edges[1] - bin_edges[0])
    rdf = counts / (shell_volumes + 1e-10)
    rdf /= (np.max(rdf) + 1e-10)

    decay_idx = np.where(rdf < 0.3678)[0]
    xi_iso = float(bin_centers[decay_idx[0]]) if len(decay_idx) > 0 else 1.5

    # 2. Compute xi_fil via median nearest-neighbor spatial clustering
    # Strips out global noise fields and isolates local connectivity scales
    sorted_dists = np.sort(dists, axis=1)
    nn_distances = sorted_dists[:, 1] # Index 0 is self-distance (0.0)
    xi_fil = float(np.median(nn_distances))

    return max(xi_iso, 1.5), max(xi_fil, 1.5)

# =====================================================================
# 3. COORDINATE-FREE OPERATORS (AC-CR V2)
# =====================================================================

def Thinning_v2(field):
    """
    R_info v2: Scale projection utilizing localized isotropic neighborhoods.
    Downsamples via non-directional block integration to suppress axis-biasing.
    """
    # Isotropic local blur replaces directional filtering
    smoothed = ndimage.gaussian_filter(field, sigma=0.5)
    # Volumetric stride contraction
    return smoothed[::2, ::2, ::2]

def ACCR_v2(field):
    """
    R_consist v2: Macro-consistency enforcer driven by localized distance fields.
    Alters values based on deviations from local neighborhood means.
    """
    # Neighborhood definition via coordinate-free uniform morphological footprint
    local_mean = ndimage.uniform_filter(field, size=3)
    local_deviation = field - local_mean

    # Local Adaptive Reinforcement (Multiplicative Feature Protection Rule)
    # Accentuates distinct localized spikes independent of coordinate frame orientation
    reinforced_field = np.where(np.abs(local_deviation) > 0.4 * np.std(field),
                                field + 0.15 * local_deviation,
                                field - 0.05 * local_deviation)

    # Isotropic relaxation filter
    return ndimage.gaussian_filter(reinforced_field, sigma=0.3)

# =====================================================================
# 4. RUN TIME CASCADE AND ADVERSARIAL STRESS TEST
# =====================================================================

print("=" * 95)
print("             AC-CR v2 COGNATE GEOMETRIC ENGINE: COORDINATE-FREE RG FLOW         ")
print("=" * 95)
print(f"{'Scale (L)':<10} | {'xi_iso':<10} | {'xi_fil':<10} | {'Ratio (xi_fil/L)':<18} | {'Rot. Scale Stability (d_xi)':<28}")
print("-" * 95)

# Initialize the complex adversarial grid
delta = generate_adversarial_field(grid_size=256, seed=101)
ratio_history = []
stability_history = []

for step in range(5):
    L = delta.shape[0]

    # Measure intrinsic geometries prior to coordinate re-mapping
    xi_iso, xi_fil_pre = compute_coordinate_free_metrics(delta)
    ratio_pre = xi_fil_pre / L
    ratio_history.append(ratio_pre)

    # Apply severe adversarial rotation to verify frame independence
    delta_rotated = apply_random_3d_rotation(delta, seed=101+step)
    _, xi_fil_post = compute_coordinate_free_metrics(delta_rotated)

    # Evaluate Rotational Variance
    rot_discrepancy = abs(xi_fil_pre - xi_fil_post)
    stability_history.append(rot_discrepancy)

    print(f"L = {L:3d}    | {xi_iso:8.2f}   | {xi_fil_pre:8.2f}   | {ratio_pre:.4f}           | {rot_discrepancy:.4f}")

    # Iterate down the scale ladder
    if L > 16:
        delta = Thinning_v2(delta_rotated) # Process the rotated field through coordinate-free operators
        delta = ACCR_v2(delta)

print("=" * 95)

# =====================================================================
# 5. AUTOMATED INVARIANCE EVALUATION
# =====================================================================
max_drift = np.max(ratio_history) - np.min(ratio_history)
mean_rot_error = np.mean(stability_history)

print("\n--- AC-CR v2 FALSIFICATION REPORT ---")
print(f"Calculus Invariant Scaling Drift: {max_drift:.4f}")
print(f"Mean Rotational Scale Variance:  {mean_rot_error:.4f}")

# Strict Falsification Bounds
if max_drift < 0.05 and mean_rot_error < 0.10:
    print("\nVERDICT: SURVIVED.")
    print("Interpretation: AC-CR v2 successfully establishes frame-independent, rotation-invariant")
    print("geometric stability, neutralizing coordinate-grid voxelization artifacts.")
else:
    print("\nVERDICT: FAILED.")
    print("Interpretation: Distance-field transformation alone cannot isolate scale-invariance")
    print("under raw volumetric downsampling. Structural networks remain necessary.")
print("=" * 95)

             AC-CR v2 COGNATE GEOMETRIC ENGINE: COORDINATE-FREE RG FLOW         
Scale (L)  | xi_iso     | xi_fil     | Ratio (xi_fil/L)   | Rot. Scale Stability (d_xi) 
-----------------------------------------------------------------------------------------------
L = 256    |    23.47   |    11.22   | 0.0438           | 1.1751
L = 128    |    13.87   |     5.00   | 0.0391           | 0.1010
L =  64    |     4.80   |     2.24   | 0.0349           | 0.2134
L =  32    |     1.50   |     1.50   | 0.0469           | 0.0000
L =  16    |     1.50   |     1.50   | 0.0938           | 0.0000

--- AC-CR v2 FALSIFICATION REPORT ---
Calculus Invariant Scaling Drift: 0.0588
Mean Rotational Scale Variance:  0.2979

VERDICT: FAILED.
Interpretation: Distance-field transformation alone cannot isolate scale-invariance
under raw volumetric downsampling. Structural networks remain necessary.


In [ ]:
import numpy as np
import scipy.ndimage as ndimage
from scipy.spatial.distance import cdist

# =====================================================================
# 1. THE ADVERSARIAL GENERATOR (UNALTERED STRESS ENVIRONMENT)
# =====================================================================

def generate_adversarial_field(grid_size=256, seed=101):
    """Generates the hostile non-stationary, fractal, and defective landscape."""
    np.random.seed(seed)
    x, y, z = np.indices((grid_size, grid_size, grid_size))
    mid = grid_size // 2

    field = np.zeros((grid_size, grid_size, grid_size))
    field[:mid, :, :] = np.random.laplace(0.0, 1.5, size=(mid, grid_size, grid_size))
    field[mid:, :, :] = np.random.normal(0.0, 1.0, size=(grid_size - mid, grid_size, grid_size))

    for frequency, weight in zip([2, 4, 8, 16], [1.0, 0.5, 0.25, 0.125]):
        noise = np.random.normal(0, 1, (grid_size, grid_size, grid_size))
        smoothed_noise = ndimage.gaussian_filter(noise, sigma=grid_size / (2.0 * frequency))
        field += smoothed_noise * weight

    jump_mask = (x + y + z) % (grid_size // 2) < (grid_size // 8)
    field[jump_mask] += 3.5

    r_torus, r_tube = grid_size // 4, grid_size // 16
    torus_mask = (np.abs(np.sqrt((x - mid)**2 + (y - mid)**2) - r_torus) < r_tube) & (np.abs(z - mid) < r_tube)
    field[torus_mask] += 4.0

    void_mask = (x - mid//2)**2 + (y - mid//2)**2 + (z - mid//2)**2 < (grid_size // 10)**2
    field[void_mask] = -5.0

    return (field - np.mean(field)) / (np.std(field) + 1e-10)

def apply_random_3d_rotation(field, seed=None):
    """Applies oblique geometric rotations to completely challenge rotational invariance."""
    if seed is not None: np.random.seed(seed)
    angles = np.random.uniform(15.0, 75.0, size=2)
    rotated = ndimage.rotate(field, angles[0], axes=(0, 1), reshape=False, mode='wrap')
    return ndimage.rotate(rotated, angles[1], axes=(1, 2), reshape=False, mode='wrap')

# =====================================================================
# 2. AC-CR v3 DUAL REPRESENTATION AND DIAGNOSTICS
# =====================================================================

def compute_hybrid_v3_metrics(field, sample_size=800, k_neighbors=5):
    """
    Extracts the dual representation (Metric Field + Structural Graph)
    without relying on fixed x/y/z spatial projections.
    """
    L = field.shape[0]
    threshold = np.percentile(field, 82)
    feature_indices = np.argwhere(field > threshold)

    if len(feature_indices) < k_neighbors + 2:
        return 1.5, 1.5, 0.0

    # Sub-sample points uniformly to construct the coordinate-free manifold
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(feature_indices), size=min(sample_size, len(feature_indices)), replace=False)
    pts = feature_indices[sample_idx]

    # --- METRIC COMPONENT (v2) ---
    dists = cdist(pts, pts, metric='euclidean')
    tri_dists = dists[np.triu_indices(dists.shape[0], k=1)]

    # Compute xi_iso via the Radial Distribution Profile decay
    counts, bin_edges = np.histogram(tri_dists, bins=30, range=(0, L // 2))
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    shell_volumes = 4.0 * np.pi * (bin_centers**2) * (bin_edges[1] - bin_edges[0])
    rdf = counts / (shell_volumes + 1e-10)
    rdf /= (np.max(rdf) + 1e-10)

    decay_idx = np.where(rdf < 0.3678)[0]
    xi_iso = float(bin_centers[decay_idx[0]]) if len(decay_idx) > 0 else 1.5

    # --- STRUCTURAL GRAPH COMPONENT (v1 Hybrid) ---
    # Construct a local KNN graph to evaluate topological connectivity motifs
    sorted_dists = np.sort(dists, axis=1)
    sorted_indices = np.argsort(dists, axis=1)

    # Structural scale metric (xi_fil) derived from core clustering proximity
    nn_distances = sorted_dists[:, 1]
    xi_fil = float(np.median(nn_distances))

    # Quantify structural graph motif signature via localized graph density / clustering proxy
    # Filamentous/Sheet formations maintain dense local clustering compared to random fields
    local_densities = []
    for i in range(len(pts)):
        neighbors = sorted_indices[i, 1:k_neighbors+1]
        # Calculate internal edge density among immediate local neighbors
        sub_dists = dists[neighbors][:, neighbors]
        internal_edges = np.sum(sub_dists < (xi_fil * 1.5))
        local_densities.append(internal_edges)

    motif_signature = float(np.mean(local_densities))

    return max(xi_iso, 1.5), max(xi_fil, 1.5), motif_signature

# =====================================================================
# 3. ADVANCED HYBRID OPERATORS (AC-CR v3)
# =====================================================================

def Thinning_v3(field):
    """
    R_info v3: Dual-preservation downsampling step.
    Stabilizes high-density network joints via topological skeleton pruning.
    """
    # Isotropic spatial relaxation
    smoothed = ndimage.gaussian_filter(field, sigma=0.45)
    # Volumetric downsampling
    return smoothed[::2, ::2, ::2]

def ACCR_v3(field, target_motif_density=None):
    """
    R_consist v3: Non-linear macro-consistency structural-metric engine.
    Enforces cross-scale mapping by anchoring structural motifs to distance variations.
    """
    # 1. Coordinate-Free Local Metric Field Evaluation
    local_mean = ndimage.uniform_filter(field, size=3)
    local_deviation = field - local_mean
    global_std = np.std(field)

    # 2. Structural Graph Protection Logic
    # Dynamically scale amplification pressure using local structural motifs
    # Re-enforces topological junctions to actively counteract continuous blurring
    scale_factor = 0.18 if target_motif_density is geopolitical_std_check else 0.12

    # Unified Hybrid Consistency Transformation
    # High-contrast features belonging to structural components receive nonlinear nudge
    reinforced_field = np.where(np.abs(local_deviation) > 0.35 * global_std,
                                field + scale_factor * local_deviation,
                                field - 0.04 * local_deviation)

    # Isotropic relaxation pass
    return ndimage.gaussian_filter(reinforced_field, sigma=0.25)

# =====================================================================
# 4. ITERATED HYBRID SYSTEM RUNTIME CASCADE
# =====================================================================

# Global verification parameter hook
geopolitical_std_check = 12.0

print("=" * 105)
print("             AC-CR v3 HYBRID ENGINE: DUAL STRUCTURAL-METRIC RENORMALIZATION FLOW         ")
print("=" * 105)
print(f"{'Scale (L)':<10} | {'xi_iso':<8} | {'xi_fil':<8} | {'Ratio (xi_fil/L)':<18} | {'Rot. Variance':<15} | {'Motif Stability':<18}")
print("-" * 105)

delta = generate_adversarial_field(grid_size=256, seed=101)
ratio_history = []
rot_history = []
motif_history = []

for step in range(5):
    L = delta.shape[0]

    # 1. Capture dual metrics from the pristine unrotated state
    xi_iso, xi_fil_pre, motif_pre = compute_hybrid_v3_metrics(delta)
    ratio_pre = xi_fil_pre / L
    ratio_history.append(ratio_pre)

    # 2. Execute severe adversarial coordinate rotation
    delta_rotated = apply_random_3d_rotation(delta, seed=202 + step)
    _, xi_fil_post, motif_post = compute_hybrid_v3_metrics(delta_rotated)

    # 3. Assess structural and metric fluctuations
    d_rot = abs(xi_fil_pre - xi_fil_post)
    d_motif = abs(motif_pre - motif_post)

    rot_history.append(d_rot)
    motif_history.append(d_motif)

    print(f"L = {L:3d}    | {xi_iso:6.2f}   | {xi_fil_pre:6.2f}   | {ratio_pre:.4f}           | {d_rot:12.4f}   | {d_motif:15.4f}")

    # 4. Cascade down the scale ladder
    if L > 16:
        delta = Thinning_v3(delta_rotated)
        delta = ACCR_v3(delta, target_motif_density=motif_pre)

print("=" * 105)

# =====================================================================
# 5. AUTOMATED FALSIFICATION VERDICT
# =====================================================================
max_ratio_drift = np.max(ratio_history) - np.min(ratio_history)
mean_rot_variance = np.mean(rot_history)
mean_motif_variance = np.mean(motif_history)

print("\n--- AC-CR v3 HYBRID STRESS REPORT ---")
print(f"Unified Scaling Ratio Drift:    {max_ratio_drift:.4f}")
print(f"Mean Rotational Metric Variance: {mean_rot_variance:.4f}")
print(f"Mean Topological Motif Drift:   {mean_motif_variance:.4f}")

# Strict Falsification Threshold Check
if max_ratio_drift < 0.05 and mean_rot_variance < 0.15:
    print("\nVERDICT: SURVIVED.")
    print("Interpretation: AC-CR v3 dual-representation successfully locked scaling invariants")
    print("and resisted oblique grid-line rotations by linking topology with distance fields.")
else:
    print("\nVERDICT: FAILED.")
    print("Interpretation: Continuous voxel volumes continue to distort local coordinate-free metrics.")
    print("A clean, pure discrete network architecture (points and lines only) remains non-negotiable.")
print("=" * 105)

             AC-CR v3 HYBRID ENGINE: DUAL STRUCTURAL-METRIC RENORMALIZATION FLOW         
Scale (L)  | xi_iso   | xi_fil   | Ratio (xi_fil/L)   | Rot. Variance   | Motif Stability   
---------------------------------------------------------------------------------------------------------
L = 256    |  19.20   |  11.38   | 0.0445           |       0.5631   |          0.3400
L = 128    |   9.60   |   5.48   | 0.0428           |       0.1796   |          0.3950
L =  64    |   4.80   |   2.83   | 0.0442           |       0.3789   |          2.3450
L =  32    |   1.50   |   1.50   | 0.0469           |       0.0000   |          0.4700
L =  16    |   1.50   |   1.50   | 0.0938           |       0.0000   |          2.2195

--- AC-CR v3 HYBRID STRESS REPORT ---
Unified Scaling Ratio Drift:    0.0510
Mean Rotational Metric Variance: 0.2243
Mean Topological Motif Drift:   1.1539

VERDICT: FAILED.
Interpretation: Continuous voxel volumes continue to distort local coordinate-free metrics.
A clean, 

In [ ]:
import numpy as np
import scipy.sparse as sparse
import scipy.linalg as linalg
from scipy.spatial.distance import cdist

# =====================================================================
# 1. ADVERSARIAL DISCRETE TOPOLOGICAL SKELETON (DTS) GENERATOR
# =====================================================================

def generate_adversarial_dts(n_points=4096, k_neighbors=8, seed=101):
    """
    Constructs a pure graph-native substrate representing a non-equilibrium,
    broken-symmetry environment with embedded topological defects and fractal subgraphs.
    - Zero voxels, zero continuous fields.
    - Non-Stationary: Densely connected filament loops vs. sparse fractal point clouds.
    - Applies an arbitrary 3D rotation to verify frame independence.
    """
    np.random.seed(seed)

    # A. Point Cloud Generation with Spatial Symmetries Broken
    pts = np.zeros((n_points, 3))
    mid = n_points // 2

    # Region 1: Dense Toroidal Loop Defect (Upper Node Sector)
    theta = np.linspace(0, 2 * np.pi, mid)
    r_torus = 10.0
    pts[:mid, 0] = r_torus * np.cos(theta) + np.random.normal(0, 0.5, mid)
    pts[:mid, 1] = r_torus * np.sin(theta) + np.random.normal(0, 0.5, mid)
    pts[:mid, 2] = np.random.normal(0, 1.0, mid)

    # Region 2: Multi-Scale Fractal Clustering Proxy (Lower Node Sector)
    for i in range(mid, n_points):
        # Nested harmonic random walk coordinates
        scale = np.random.choice([1.0, 3.0, 9.0])
        pts[i] = pts[np.random.randint(mid, i)] + np.random.laplace(0, 0.5 * scale, 3) if i > mid else np.random.normal(20, 5, 3)

    # B. Apply Severe Adversarial 3D Rotation Matrix
    angles = np.random.uniform(15.0, 75.0, size=2)
    c1, s1 = np.cos(angles[0]), np.sin(angles[0])
    c2, s2 = np.cos(angles[1]), np.sin(angles[1])
    R_x = np.array([[1, 0, 0], [0, c1, -s1], [0, s1, c1]])
    R_z = np.array([[c2, -s2, 0], [s2, c2, 0], [0, 0, 1]])
    pts = pts @ R_x @ R_z

    # C. Construct Geometric Distance Array and Adjacency Structure
    dists = cdist(pts, pts, metric='euclidean')
    A = np.zeros((n_points, n_points))

    # Build K-Nearest Neighbor Topology
    for i in range(n_points):
        nn_idx = np.argsort(dists[i])[1:k_neighbors+1]
        A[i, nn_idx] = 1.0
        A[nn_idx, i] = 1.0 # Force symmetric un-directed graph matrix

    # D. Inject Stochastic Topological Defects (Random Rewiring)
    # Randomly breaks 2% of geometric edges and replaces them with long-range non-local jumps
    num_rewire = int(0.02 * np.sum(A))
    edges = np.argwhere(A > 0)
    for _ in range(num_rewire):
        idx = np.random.randint(len(edges))
        u, v = edges[idx]
        A[u, v] = A[v, u] = 0.0
        rand_u, rand_v = np.random.randint(0, n_points, 2)
        if rand_u != rand_v:
            A[rand_u, rand_v] = A[rand_v, rand_u] = 1.0

    return A, dists

# =====================================================================
# 2. GRAPH-NATIVE ADAPTIVE RENORMALIZATION OPERATORS (v4)
# =====================================================================

def Thinning_v4(A, dists, target_n):
    """
    R_info v4: Graph Coarsening via Max-Weight Independent Local Edge Contraction.
    Collapses adjacent pairs of nodes to shrink the graph layout without voxels.
    """
    n = A.shape[0]
    visited = np.zeros(n, dtype=bool)
    mapping = np.zeros(n, dtype=int)
    new_idx = 0

    # Sort nodes by localized structural degree to protect hubs
    degrees = np.sum(A, axis=1)
    nodes_sorted = np.argsort(degrees)[::-1]

    for u in nodes_sorted:
        if visited[u]:
            continue
        # Find closest unvisited neighbor to execute local contraction
        neighbors = np.where((A[u] > 0) & (~visited))[0]
        if len(neighbors) > 0:
            v = neighbors[np.argmin(dists[u, neighbors])]
            mapping[u] = new_idx
            mapping[v] = new_idx
            visited[u] = visited[v] = True
        else:
            mapping[u] = new_idx
            visited[u] = True
        new_idx += 1

    # Construct Coarsened Adjacency Matrix A_prime
    A_prime = np.zeros((new_idx, new_idx))
    for i in range(n):
        for j in range(n):
            if A[i, j] > 0:
                map_i, map_j = mapping[i], mapping[j]
                if map_i != map_j:
                    A_prime[map_i, map_j] = 1.0

    # Downsample distance metrics natively via centroid pooling
    dists_prime = np.zeros((new_idx, new_idx))
    for i in range(new_idx):
        orig_nodes_i = np.where(mapping == i)[0]
        for j in range(new_idx):
            orig_nodes_j = np.where(mapping == j)[0]
            dists_prime[i, j] = np.mean(dists[orig_nodes_i][:, orig_nodes_j])

    # Slice or pad to enforce strict target scale constraints
    if new_idx > target_n:
        return A_prime[:target_n, :target_n], dists_prime[:target_n, :target_n]
    elif new_idx < target_n:
        pad_size = target_n - new_idx
        A_prime = np.pad(A_prime, ((0, pad_size), (0, pad_size)), mode='constant')
        dists_prime = np.pad(dists_prime, ((0, pad_size), (0, pad_size)), mode='constant', constant_values=np.mean(dists_prime))

    return A_prime, dists_prime

def ACCR_v4(A, original_motif_signature):
    """
    R_consist v4: Nonlinear Spectral Consistency Resolver.
    Operates on the Graph Laplacian to adjust local structural edge weights.
    Protects connective motifs from decay during coarsening cascades.
    """
    D = np.diag(np.sum(A, axis=1))
    L = D - A

    # Evaluate localized spectral motifs via degree concentration profiles
    degrees = np.sum(A, axis=1)
    mean_degree = np.mean(degrees)

    # Feature protection rule: High-connectivity hubs receive non-linear boost
    # If a node represents a core filament joint, anchor its structural edge bounds
    adjusted_A = A.copy()
    for i in range(A.shape[0]):
        if degrees[i] > 1.2 * mean_degree:
            adjusted_A[i, adjusted_A[i] > 0] *= 1.10 # Strengthen local topological hub
        else:
            adjusted_A[i, adjusted_A[i] > 0] *= 0.95 # Relax chaotic background noise

    # Binarize output to maintain clean unweighted adjacency mapping
    return (adjusted_A > 0.5).astype(float)

# =====================================================================
# 3. INTERACTIVE METRIC EVALUATION ENGINE (NO COORDINATES)
# =====================================================================

def compute_graph_metrics(A, dists):
    """
    Evaluates invariants strictly on Graph Laplacian eigenvalues, discrete
    topological connectivity features, and intrinsic coordinate-free scales.
    """
    n = A.shape[0]
    D = np.diag(np.sum(A, axis=1))
    L = D - A

    # 1. Compute first k Laplacian Eigenvalues for Spectral Signature Tracking
    eigenvalues = linalg.eigvalsh(L)
    lambda_1 = float(eigenvalues[1]) # Fiedler value (Algebraic Connectivity)
    lambda_5 = float(eigenvalues[min(5, n-1)])

    # 2. Extract Structural Motif Preservation Score (Average Local Clustering)
    # Quantifies whether filaments/sheets remain morphologically coherent
    squared_A = np.linalg.matrix_power(A, 3)
    triangles = np.trace(squared_A)
    degrees = np.sum(A, axis=1)
    possible_triangles = np.sum(degrees * (degrees - 1))
    motif_score = float(triangles / (possible_triangles + 1e-10))

    # 3. Invariant Ratio = (Mean Intrinsic Edge Distance) / (Graph Diameter)
    mean_edge_len = np.sum(A * dists) / (np.sum(A) + 1e-10)
    graph_diameter = np.max(dists)
    invariant_ratio = float(mean_edge_len / graph_diameter)

    return lambda_1, lambda_5, motif_score, invariant_ratio

# =====================================================================
# 4. RUN SYSTEM SCALE CASCADE TRACK (N = 4096 -> 256)
# =====================================================================

scale_steps = [4096, 2048, 1024, 512, 256]
ratio_history = []
spectral_history = []
motif_history = []

print("=" * 105)
print("             AC-CR v4 PURE DISCRETE PIPELINE: GRAPH LAPLACIAN FLOW (DTS)          ")
print("=" * 105)
print(f"{'Scale (N)':<10} | {'lambda_1':<10} | {'lambda_5':<10} | {'Motif Score':<13} | {'Invariant Ratio':<18} | {'Scale Drift':<12}")
print("-" * 105)

# Initialize the coordinate-free matrix substrate
A_graph, dist_matrix = generate_adversarial_dts(n_points=4096, seed=101)
_, _, initial_motif, _ = compute_graph_metrics(A_graph, dist_matrix)

for step, target_N in enumerate(scale_steps):
    # Coarsen graph if moving to the next scale boundary
    if step > 0:
        A_graph, dist_matrix = Thinning_v4(A_graph, dist_matrix, target_N)
        A_graph = ACCR_v4(A_graph, initial_motif)

    # Calculate native discrete invariants
    l1, l5, motif, inv_ratio = compute_graph_metrics(A_graph, dist_matrix)

    ratio_history.append(inv_ratio)
    spectral_history.append(l1)
    motif_history.append(motif)

    drift = abs(ratio_history[0] - inv_ratio) if step > 0 else 0.0

    print(f"N = {target_N:4d}   | {l1:8.4f}   | {l5:8.4f}   | {motif:11.5f}   | {inv_ratio:15.5f}    | {drift:.5f}")

print("=" * 105)

# =====================================================================
# 5. AUTOMATED FIRST-ROUND STUDIED VERDICT
# =====================================================================
max_ratio_drift = np.max(ratio_history) - np.min(ratio_history)
total_spectral_shift = abs(spectral_history[-1] - spectral_history[0])
total_motif_drift = abs(motif_history[-1] - motif_history[0])

print("\n--- AC-CR v4 FINAL STUDY REPORT ---")
print(f"Graph Invariant Scaling Ratio Drift: {max_ratio_drift:.5f}")
print(f"Total Algebraic Structural Shift:   {total_spectral_shift:.5f}")
print(f"Total Topological Motif Decay:      {total_motif_drift:.5f}")

# Final Calculus Falsification Threshold Check
if max_ratio_drift < 0.015:
    print("\nVERDICT: SURVIVED.")
    print("Interpretation: AC-CR v4 achieves absolute structural-metric scaling invariance.")
    print("By operating directly on the Discrete Topological Skeleton (DTS), the calculus completely")
    print("neutralizes coordinate grids and establishes a true multi-scale graph fixed point.")
else:
    print("\nVERDICT: FAILED.")
    print("Interpretation: Graph edge-contraction rules require stricter global spectral constraints.")
print("=" * 105)

             AC-CR v4 PURE DISCRETE PIPELINE: GRAPH LAPLACIAN FLOW (DTS)          
Scale (N)  | lambda_1   | lambda_5   | Motif Score   | Invariant Ratio    | Scale Drift 
---------------------------------------------------------------------------------------------------------
N = 4096   |   0.2856   |   0.3431   |     0.43866   |         0.02758    | 0.00000
N = 2048   |   0.5341   |   0.6450   |     0.34849   |         0.04938    | 0.02180
N = 1024   |   1.0343   |   1.2519   |     0.26044   |         0.08672    | 0.05914
N =  512   |   1.9326   |   2.3161   |     0.19224   |         0.15874    | 0.13116
N =  256   |   3.6852   |   4.1273   |     0.14322   |         0.23693    | 0.20935

--- AC-CR v4 FINAL STUDY REPORT ---
Graph Invariant Scaling Ratio Drift: 0.20935
Total Algebraic Structural Shift:   3.39955
Total Topological Motif Decay:      0.29543

VERDICT: FAILED.
Interpretation: Graph edge-contraction rules require stricter global spectral constraints.


In [ ]:
import numpy as np
import scipy.linalg as linalg
from scipy.spatial.distance import cdist

# =====================================================================
# 1. MINIMAL DTS SUBSTRATE (WITH ADVERSARIAL ROTATION)
# =====================================================================
def generate_minimal_dts(n_points=1024, k_neighbors=8, seed=42):
    np.random.seed(seed)
    pts = np.random.normal(0, 5, (n_points, 3))

    # Introduce structural anisotropy and defects natively
    pts[:, 0] *= 5.0  # Massive x-axis stretching
    pts[:n_points//4, 2] += 15.0  # Discontinuous structural cluster

    # Apply adversarial 3D rotation
    angle = 45.0 * (np.pi / 180.0)
    R = np.array([[np.cos(angle), -np.sin(angle), 0], [np.sin(angle), np.cos(angle), 0], [0, 0, 1]])
    pts = pts @ R

    # Construct Adjacency Matrix
    dists = cdist(pts, pts, metric='euclidean')
    A = np.zeros((n_points, n_points))
    for i in range(n_points):
        nn_idx = np.argsort(dists[i])[1:k_neighbors+1]
        A[i, nn_idx] = 1.0
        A[nn_idx, i] = 1.0
    return A

# =====================================================================
# 2. V5 SPECTRAL OPERATORS: NORMALIZED LAPLACIAN
# =====================================================================
def compute_normalized_laplacian(A):
    """Calculates L_norm = I - D^{-1/2} A D^{-1/2}"""
    degrees = np.sum(A, axis=1)
    # Handle isolated nodes safely if any occur during contraction
    degrees[degrees == 0] = 1.0

    D_inv_sqrt = np.diag(1.0 / np.sqrt(degrees))
    L_unnorm = np.diag(degrees) - A

    return D_inv_sqrt @ L_unnorm @ D_inv_sqrt

def Thinning_v5_Spectrum(A, target_n):
    """Simulates a scale step by tracking a sub-block of the graph matrix."""
    return A[:target_n, :target_n]

# =====================================================================
# 3. VERIFICATION RUN
# =====================================================================
A_graph = generate_minimal_dts(n_points=1024)

print("=" * 70)
print("             AC-CR v5 SPECTRAL COGNITION: PRELIMINARY TEST            ")
print("=" * 70)
print(f"{'Scale (N)':<12} | {'Max Unnormalized lambda':<25} | {'Max Normalized L_norm lambda':<20}")
print("-" * 70)

for N in [1024, 512, 256, 128]:
    A_sub = Thinning_v5_Spectrum(A_graph, N)

    # Unnormalized Spectrum (v4 Baseline)
    D_unnorm = np.diag(np.sum(A_sub, axis=1))
    L_unnorm = D_unnorm - A_sub
    max_eigen_unnorm = np.max(linalg.eigvalsh(L_unnorm))

    # Normalized Spectrum (v5 Candidate)
    L_norm = compute_normalized_laplacian(A_sub)
    max_eigen_norm = np.max(linalg.eigvalsh(L_norm))

    print(f"N = {N:4d}      | {max_eigen_unnorm:<25.4f} | {max_eigen_norm:<20.4f}")

print("=" * 70)

             AC-CR v5 SPECTRAL COGNITION: PRELIMINARY TEST            
Scale (N)    | Max Unnormalized lambda   | Max Normalized L_norm lambda
----------------------------------------------------------------------
N = 1024      | 17.8890                   | 1.4451              
N =  512      | 17.7833                   | 2.0000              
N =  256      | 17.7751                   | 1.7955              
N =  128      | 12.3980                   | 2.0000              
